<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_14_Mini_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch
import torch.nn as nn
import math

In [15]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0, \
            "Embedding dimension must be divisible by number of heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Query, Key, Value projections
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # Final output projection
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        # Generate Q, K, V
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Split into multiple heads
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        # Move head dimension forward
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        # Scale
        scores = scores / math.sqrt(self.head_dim)

        # Softmax
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Context vectors
        context = torch.matmul(
            attention_weights,
            V
        )

        # Restore dimensions
        context = context.transpose(1, 2)

        context = context.reshape(
            batch_size,
            seq_len,
            self.embed_dim
        )

        # Final projection
        output = self.fc_out(context)

        return output

In [16]:
class Transformer(nn.Module):
  def __init__(self, embed_dim,num_heads):
      super().__init__()

      self.attention=MultiHeadAttention(
          embed_dim,
          num_heads
      )

      self.norm1=nn.LayerNorm(embed_dim)

      self.ffn=nn.Sequential(
          nn.Linear(embed_dim,embed_dim*4),
          nn.ReLU(),
          nn.Linear(embed_dim*4,embed_dim)
      )

      self.norm2=nn.LayerNorm(embed_dim)

  def forward(self,x):
    attention_output=self.attention(x)

    #residuaal + Norm

    x=self.norm1(x + attention_output)

    ffn_output=self.ffn(x)

    x=self.norm2(x+ffn_output)

    return x

In [17]:
#Mini GPT

In [18]:
class MiniGpt(nn.Module):
  def __init__(self,vocab_size,embed_dim,num_heads,num_layers):
     super().__init__()

     self.embedding=nn.Embedding(vocab_size,embed_dim)

     self.blocks=nn.Sequential(
         *[
             Transformer(embed_dim,num_heads)
             for _ in range(num_layers)
         ]
     )
     self.fc=nn.Linear(
         embed_dim,
         vocab_size
     )

  def forward(self,x):
    x=self.embedding(x)

    x=self.blocks(x)

    logits=self.fc(x)

    return logits



In [19]:
vocab = {
    "i":0,
    "love":1,
    "ai":2,
    "python":3,
    "data":4
}

In [20]:
tokens = torch.tensor([
    [0,1,2]
])

In [21]:
model = MiniGpt(
    vocab_size=5,
    embed_dim=8,
    num_heads=2,
    num_layers=2
)

In [22]:
output = model(tokens)

print(output.shape)

torch.Size([1, 3, 5])


In [23]:
print(output)

tensor([[[-1.4099, -0.0786, -0.7221,  0.5344, -0.4137],
         [ 0.6809,  0.0834,  0.5966, -0.9657,  0.4361],
         [-0.7306, -0.9241, -1.0319,  0.0425,  0.0440]]],
       grad_fn=<ViewBackward0>)


**What GPT Does Internally**

Input
↓
Predict next word
↓
Append
↓
Predict again
↓
Append
↓
Repeat

Exactly how ChatGPT, Claude, Gemini, Llama generate text.

### **Interview-Level Summary**

**Why Embedding?**
Converts token IDs into dense semantic vectors.

**Why Positional Encoding?**
Attention has no sense of order, positional information is added.

**Why Attention?**
Allows every token to look at every other token.

**Why Multi-Head Attention?**
Different heads learn different relationships.

**Why FFN?**
Processes information gathered by attention.

**Why Residual Connections?**
Prevent information loss and help gradient flow.

**Why LayerNorm?**
Stabilizes training.

**Why Linear + Softmax?**
Converts hidden representations into probabilities over the vocabulary.

**How GPT Generates Text?**
Predict one token → append → predict again → repeat.
